# 1. Construção da base analítica por UF

Este notebook é responsável exclusivamente por ler, inspecionar, preparar e validar os **Indicadores de Fluxo da Educação Superior do INEP** no nível de Unidade da Federação (UF).

Ao final, será gerada a base `base_modelo_uf.csv`, utilizada pelos notebooks de protocolo de validação e comparação de modelos.

## 1.1 Configuração e acesso aos dados

In [112]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()

# Procura a raiz do projeto subindo pelos diretórios
while (
    PROJECT_ROOT != PROJECT_ROOT.parent
    and not (PROJECT_ROOT / "src").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(
        "Não foi possível localizar a pasta 'src'. "
        "Verifique se o notebook está sendo executado dentro do projeto."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"src encontrado: {(PROJECT_ROOT / 'src').exists()}")
print(f"preprocessing.py encontrado: {(PROJECT_ROOT / 'src' / 'preprocessing.py').exists()}")

Raiz do projeto: /home/sara/Documentos/tcc-evasao-ensino-superior
src encontrado: True
preprocessing.py encontrado: True


In [113]:
from src.preprocessing import preparar_indicador

In [114]:
ARQUIVO_UF = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "INDIC_UF_2010_2024.xlsx"
)

print(f"Base: {ARQUIVO_UF}")

Base: /home/sara/Documentos/tcc-evasao-ensino-superior/data/raw/INDIC_UF_2010_2024.xlsx


In [115]:
xls = pd.ExcelFile(ARQUIVO_UF)

ab_as = xls.sheet_names
print("Abas encontradas:")
for aba in ab_as:
    print(f"- {aba}")

abas_esperadas = {
    "TX_EVASAO",
    "TX_CONCLUSAO",
    "TX_RETENCAO",
    "TX_PERMANENCIA"
}

assert set(ab_as) == abas_esperadas, (
    f"As abas encontradas não correspondem ao esperado: {ab_as}"
)

Abas encontradas:
- TX_EVASAO
- TX_CONCLUSAO
- TX_RETENCAO
- TX_PERMANENCIA


## 1.2 Inspeção da estrutura original

In [116]:
# A planilha possui cabeçalhos em múltiplas linhas.
# A inspeção abaixo preserva a estrutura original antes da transformação.
estrutura_abas = {}
for aba in ab_as:
    df_raw = pd.read_excel(
        ARQUIVO_UF,
        sheet_name=aba,
        header=None
    )
    estrutura_abas[aba] = df_raw.shape

pd.DataFrame.from_dict(
    estrutura_abas,
    orient="index",
    columns=["linhas", "colunas"]
)

,linhas,colunas
TX_EVASAO,389,16
TX_CONCLUSAO,227,16
TX_RETENCAO,227,14
TX_PERMANENCIA,227,16


In [117]:
df_evasao_raw = pd.read_excel(
    ARQUIVO_UF,
    sheet_name="TX_EVASAO",
    header=None
)

# Visualização compacta do cabeçalho e dos primeiros registros reais.
df_evasao_raw.iloc[5:12, :].T

,5,6,7,8,9,10,11
0,Ano Fluxo,NaN,NaN,2010-2011,NaN,NaN,NaN
1,Unidade da Federação,NaN,NaN,Acre,Alagoas,Amapá,Amazonas
2,Total,NaN,NaN,8.4,10.5,15.9,11.7
3,Sexo,Feminino,NaN,7.9,10.1,14.2,10.9
4,NaN,Masculino,NaN,9.1,11.1,18.1,12.7
5,Preto/Pardo/Indígena (PPI),Sim,NaN,8.5,3.3,14.4,7.6
6,NaN,Não 1,NaN,4.7,3.2,17.3,14.2
7,Faixa de Idade,Até 19 anos,NaN,5.3,7.7,10.8,5.6
8,NaN,20 a 22 anos,NaN,6.5,7.8,14,6.6
9,NaN,23 e 24 anos,NaN,8.8,9.7,16.7,10.4


## 1.3 Preparação dos indicadores

In [118]:
df_evasao = preparar_indicador(
    ARQUIVO_UF,
    "TX_EVASAO",
    "evasao",
    possui_deficiencia=True
)

df_conclusao = preparar_indicador(
    ARQUIVO_UF,
    "TX_CONCLUSAO",
    "conclusao",
    possui_deficiencia=True
)

df_retencao = preparar_indicador(
    ARQUIVO_UF,
    "TX_RETENCAO",
    "retencao",
    possui_deficiencia=False
)

df_permanencia = preparar_indicador(
    ARQUIVO_UF,
    "TX_PERMANENCIA",
    "permanencia",
    possui_deficiencia=True
)

In [119]:
resumo_indicadores = []

for nome, df in {
    "Evasão": df_evasao,
    "Conclusão": df_conclusao,
    "Retenção": df_retencao,
    "Permanência": df_permanencia
}.items():
    resumo_indicadores.append({
        "indicador": nome,
        "linhas": len(df),
        "colunas": len(df.columns),
        "periodos": df["ano_fluxo"].nunique(),
        "ufs": df["uf"].nunique(),
        "duplicidades_uf_periodo": df.duplicated(["ano_fluxo", "uf"]).sum(),
        "valores_ausentes": int(df.isna().sum().sum())
    })

pd.DataFrame(resumo_indicadores)

,indicador,linhas,colunas,periodos,ufs,duplicidades_uf_periodo,valores_ausentes
0,Evasão,378,16,14,27,0,0
1,Conclusão,216,16,8,27,0,0
2,Retenção,216,14,8,27,0,0
3,Permanência,216,16,8,27,0,0


## 1.4 Construção da base temporal

In [120]:
df_total_evasao = df_evasao[
    ["ano_fluxo", "uf", "evasao_total"]
].copy()

df_total_conclusao = df_conclusao[
    ["ano_fluxo", "uf", "conclusao_total"]
].copy()

df_total_retencao = df_retencao[
    ["ano_fluxo", "uf", "retencao_total"]
].copy()

df_total_permanencia = df_permanencia[
    ["ano_fluxo", "uf", "permanencia_total"]
].copy()

df_indicadores = (
    df_total_evasao
    .merge(df_total_conclusao, on=["ano_fluxo", "uf"], how="inner")
    .merge(df_total_retencao, on=["ano_fluxo", "uf"], how="inner")
    .merge(df_total_permanencia, on=["ano_fluxo", "uf"], how="inner")
)

print("Base combinada:", df_indicadores.shape)

Base combinada: (216, 6)


In [121]:
df_indicadores = (
    df_indicadores
    .sort_values(["uf", "ano_fluxo"])
    .reset_index(drop=True)
)

colunas_defasadas = [
    "evasao_total",
    "conclusao_total",
    "retencao_total",
    "permanencia_total"
]

for coluna in colunas_defasadas:
    df_indicadores[f"{coluna}_t_1"] = (
        df_indicadores
        .groupby("uf")[coluna]
        .shift(1)
    )

df_indicadores.head(10)

,ano_fluxo,uf,evasao_total,conclusao_total,retencao_total,permanencia_total,evasao_total_t_1,conclusao_total_t_1,retencao_total_t_1,permanencia_total_t_1
0,2016-2017,Acre,13.9,77.6,25.4,80.2,NaN,NaN,NaN,NaN
1,2017-2018,Acre,13.9,76.2,23.4,79.9,13.9,77.6,25.4,80.2
2,2018-2019,Acre,13.2,75.3,22.7,81.3,13.9,76.2,23.4,79.9
3,2019-2020,Acre,11.5,68.2,36.3,83.9,13.2,75.3,22.7,81.3
4,2020-2021,Acre,14.3,59.8,40.2,80.8,11.5,68.2,36.3,83.9
5,2021-2022,Acre,15.2,68.4,33.3,78.4,14.3,59.8,40.2,80.8
6,2022-2023,Acre,17.6,63.7,33.7,74.7,15.2,68.4,33.3,78.4
7,2023-2024,Acre,17.6,61.6,36.9,75.9,17.6,63.7,33.7,74.7
8,2016-2017,Alagoas,12.0,59.5,37.3,84.4,NaN,NaN,NaN,NaN
9,2017-2018,Alagoas,14.1,58.6,39.6,80.7,12.0,59.5,37.3,84.4


### Tratamento da primeira observação de cada UF

As variáveis `t-1` não existem para o primeiro período disponível de cada UF. Portanto, esses `NaN` são esperados e representam a ausência de um período anterior, e não uma falha ou ausência de informação na fonte.

In [122]:
colunas_t1 = [
    "evasao_total_t_1",
    "conclusao_total_t_1",
    "retencao_total_t_1",
    "permanencia_total_t_1"
]

# Verifica que os únicos NaN de t-1 são os 27 primeiros registros,
# um para cada UF.
assert df_indicadores[colunas_t1].isna().sum().eq(27).all()

# Verifica que não existem lacunas intermediárias na trajetória das UFs.
lacunas_temporais = (
    df_indicadores
    .groupby("uf")["ano_fluxo"]
    .agg(["count", "nunique"])
    .query("count != 8 or nunique != 8")
)

assert lacunas_temporais.empty

print("27 observações iniciais apresentam NaN nas variáveis t-1, uma por UF.")
print("Nenhuma lacuna temporal intermediária foi identificada.")

27 observações iniciais apresentam NaN nas variáveis t-1, uma por UF.
Nenhuma lacuna temporal intermediária foi identificada.


In [123]:
df_modelo_base = df_indicadores.dropna(
    subset=colunas_t1
).copy()

df_modelo_base = df_modelo_base.rename(columns={
    "evasao_total": "evasao_t",
    "evasao_total_t_1": "evasao_t_1",
    "conclusao_total": "conclusao_t",
    "conclusao_total_t_1": "conclusao_t_1",
    "retencao_total": "retencao_t",
    "retencao_total_t_1": "retencao_t_1",
    "permanencia_total": "permanencia_t",
    "permanencia_total_t_1": "permanencia_t_1"
})

df_modelo_base = df_modelo_base.reset_index(drop=True)

print("Base final para modelagem:", df_modelo_base.shape)
df_modelo_base.head()

Base final para modelagem: (189, 10)


,ano_fluxo,uf,evasao_t,conclusao_t,retencao_t,permanencia_t,evasao_t_1,conclusao_t_1,retencao_t_1,permanencia_t_1
0,2017-2018,Acre,13.9,76.2,23.4,79.9,13.9,77.6,25.4,80.2
1,2018-2019,Acre,13.2,75.3,22.7,81.3,13.9,76.2,23.4,79.9
2,2019-2020,Acre,11.5,68.2,36.3,83.9,13.2,75.3,22.7,81.3
3,2020-2021,Acre,14.3,59.8,40.2,80.8,11.5,68.2,36.3,83.9
4,2021-2022,Acre,15.2,68.4,33.3,78.4,14.3,59.8,40.2,80.8


## 1.5 Validações finais da base

In [124]:
assert df_modelo_base.shape == (189, 10)
assert df_modelo_base.duplicated(["ano_fluxo", "uf"]).sum() == 0
assert df_modelo_base.isna().sum().sum() == 0
assert df_modelo_base["uf"].nunique() == 27
assert df_modelo_base["ano_fluxo"].nunique() == 7

print("Validações finais aprovadas.")

Validações finais aprovadas.


In [125]:
df_modelo_base[[
    "uf",
    "ano_fluxo",
    "evasao_t",
    "evasao_t_1",
    "conclusao_t_1",
    "retencao_t_1",
    "permanencia_t_1"
]].head(10)

,uf,ano_fluxo,evasao_t,evasao_t_1,conclusao_t_1,retencao_t_1,permanencia_t_1
0,Acre,2017-2018,13.9,13.9,77.6,25.4,80.2
1,Acre,2018-2019,13.2,13.9,76.2,23.4,79.9
2,Acre,2019-2020,11.5,13.2,75.3,22.7,81.3
3,Acre,2020-2021,14.3,11.5,68.2,36.3,83.9
4,Acre,2021-2022,15.2,14.3,59.8,40.2,80.8
5,Acre,2022-2023,17.6,15.2,68.4,33.3,78.4
6,Acre,2023-2024,17.6,17.6,63.7,33.7,74.7
7,Alagoas,2017-2018,14.1,12.0,59.5,37.3,84.4
8,Alagoas,2018-2019,15.9,14.1,58.6,39.6,80.7
9,Alagoas,2019-2020,14.2,15.9,60.0,35.7,78.5


## 1.6 Exportação da base processada

In [126]:
CAMINHO_BASE_PROCESSADA = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "base_modelo_uf.csv"
)

CAMINHO_BASE_PROCESSADA.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_modelo_base.to_csv(
    CAMINHO_BASE_PROCESSADA,
    index=False
)

print(f"Base processada salva em: {CAMINHO_BASE_PROCESSADA}")

Base processada salva em: /home/sara/Documentos/tcc-evasao-ensino-superior/data/processed/base_modelo_uf.csv


In [127]:
df_verificacao = pd.read_csv(
    CAMINHO_BASE_PROCESSADA
)

assert df_verificacao.shape == df_modelo_base.shape
assert list(df_verificacao.columns) == list(df_modelo_base.columns)

print("Arquivo CSV verificado com sucesso.")
print("Shape:", df_verificacao.shape)
df_verificacao.head()

Arquivo CSV verificado com sucesso.
Shape: (189, 10)


,ano_fluxo,uf,evasao_t,conclusao_t,retencao_t,permanencia_t,evasao_t_1,conclusao_t_1,retencao_t_1,permanencia_t_1
0,2017-2018,Acre,13.9,76.2,23.4,79.9,13.9,77.6,25.4,80.2
1,2018-2019,Acre,13.2,75.3,22.7,81.3,13.9,76.2,23.4,79.9
2,2019-2020,Acre,11.5,68.2,36.3,83.9,13.2,75.3,22.7,81.3
3,2020-2021,Acre,14.3,59.8,40.2,80.8,11.5,68.2,36.3,83.9
4,2021-2022,Acre,15.2,68.4,33.3,78.4,14.3,59.8,40.2,80.8
